## Imports

In [1]:
import os
import random
import torch
import numpy as np

from fairface_vit       import FairFaceViT
from dataset.dataloader import get_dataset, get_dataloaders
from dataset.transforms import get_train_transform, get_age_transform, get_val_transform
from training.trainer   import evaluate_loop
from training.losses    import get_age_weights, get_race_weights, get_loss_function

/home/ashkanrn/01-Project/University/Deep-Learning/DL-project/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Seed

In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Model selection

In [3]:
# change ONLY this + run_version to evaluate a different trained model
model_name = "clip"  # "clip" | "dinov2" | "siglip"
run_version = "v2"
 
if model_name == "clip":
    from model.clip import model as backbone_model, processor as backbone_processor
 
elif model_name == "dinov2":
    from model.dinov2 import model as backbone_model, processor as backbone_processor
 
elif model_name == "siglip":
    from model.siglip import model as backbone_model, processor as backbone_processor
 
else:
    raise ValueError(f"Unknown model_name: {model_name}")
 
RUN_NAMES = {
    "clip"  : "clip-vit-base-patch16",
    "dinov2": "dinov2-vit-base",
    "siglip": "siglip-vit-base-patch16",
}
 
run_name = RUN_NAMES[model_name]
 
checkpoint_dir  = f"../checkpoints/{model_name}/{run_version}"
best_heads_path = f"{checkpoint_dir}/{model_name}-best-heads.pt"
 
if not os.path.exists(best_heads_path):
    raise FileNotFoundError(f"No best-heads checkpoint found at {best_heads_path}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 31075.86it/s]
[transformers] CLIPVisionModel LOAD REPORT from: ../models/clip-vit-base-patch16
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_

## Hyperparameters

In [4]:
# # must match the values used to build the head architecture during training
# age_dropout1   = 0.15
# age_hidden_dim = 384
# batch_size     = 20
 
# # only used to instantiate the loss functions evaluate_loop expects;
# # doesn't affect reported accuracy/f1/mae metrics


## Model and device

In [5]:
fairface_model = FairFaceViT(backbone_model)
device = "cuda" if torch.cuda.is_available() else "cpu"
fairface_model.to(device)

## Load best heads checkpoint
checkpoint = torch.load(
    best_heads_path,
    map_location=device,
    weights_only=False
)
 
fairface_model.gender.load_state_dict(checkpoint["gender_head"])
fairface_model.age.load_state_dict(checkpoint["age_head"])
fairface_model.race.load_state_dict(checkpoint["race_head"])
 
fairface_model.eval()
 
print(
    f"Loaded {model_name}-best-heads.pt | "
    f"trained epoch={checkpoint['epoch']} | "
    f"best val weighted F1={checkpoint['best_acc']:.4f}"
)

Loaded clip-best-heads.pt | trained epoch=3 | best val weighted F1=0.7303


## Dataset and dataloaders

In [6]:
train_set, val_set, test_set = get_dataset(
    get_train_transform(backbone_processor),
    get_age_transform(backbone_processor),
    get_val_transform(backbone_processor),
)

_, val_dataloder, test_loader = get_dataloaders(
    train_set,
    val_set,
    test_set
)

load_dataset: 8.57s
extract splits: 0.00s
prepare indices: 0.00s
train_test_split: 0.19s
select(): 0.02s


## Loss functions

In [7]:
age_labels = np.array(train_set.dataset["age"])
race_labels = np.array(train_set.dataset["race"])

age_weights = get_age_weights(age_labels, num_classes=9, device=device)
race_weights = get_race_weights(race_labels, num_classes=7, device=device)

loss_funcs = get_loss_function(age_weights, race_weights)

## Final evaluation on held-out test set

In [ ]:
LOSS_WEIGHT = {
    "gender": 1,
    "age":    1,
    "race":   1
}

test_loss, test_task_loss, test_metrics, test_subgroup_metrics = evaluate_loop(
    val_dataloder, fairface_model, loss_funcs, LOSS_WEIGHT, device, epoch=1, epochs=1,use_amp=False
)
 
avg_accuracy = (
    test_metrics["gender"]["accuracy"]
    +
    test_metrics["age"]["accuracy"]
    +
    test_metrics["race"]["accuracy"]
) / 3
task_weighted_f1 = (0.25 * test_metrics['gender']['f1']) + (0.4 * test_metrics['age']['f1']) + (0.35 * test_metrics['race']['f1'])
 
print("\n=== TEST SET RESULTS ===")
print(f"model: {run_name} | run_version: {run_version}\n")
print(f"avg loss: {test_loss:.4f}")
print(f"avg accuracy (3-task mean): {avg_accuracy:.4f}")
print(f"task weighted f1: {task_weighted_f1:.4f}\n")

for task, metrics in test_metrics.items():
    print(f"\n{task}: ")

    for metric, value in metrics.items():
        print(f"{metric:}, {value:.f4}")


print("\n--- subgroup metrics ---")
for group_name, values in test_subgroup_metrics.items():
    print(f"{group_name}:")
    for subgroup, acc in values.items():
        print(f"  {subgroup}: {acc:.3f}")

-VALIDATION
 avg loss: 2.3547
 gender: acc=0.956  f1=0.953
 age:    acc=0.613  f1=0.590   mae=0.424
 race:   acc=0.733  f1=0.729

=== TEST SET RESULTS ===
model: clip-vit-base-patch16 | run_version: v2
avg loss: 2.3547
avg accuracy (3-task mean): 0.7672
task weighted f1: 0.6929
gender
accuracy 0.9558152273142231
precision 0.9609621451104101
recall 0.9445736434108527
f1 0.952697419859265
age
accuracy 0.6131093664414826
precision 0.5885703667167788
recall 0.6012286371204062
f1 0.5900799350783171
mse 0.5198101150264743
rmse 0.7209785815310149
mae 0.4237721380317692
race
accuracy 0.7327003834215812
precision 0.731464710370832
recall 0.7291717328428652
f1 0.7291207733052081

--- subgroup metrics ---
gender by race:
  0: 0.958
  1: 0.960
  2: 0.915
  3: 0.962
  4: 0.977
  5: 0.967
  6: 0.955
gender by age:
  0: 0.869
  1: 0.869
  2: 0.929
  3: 0.972
  4: 0.981
  5: 0.981
  6: 0.970
  7: 0.981
  8: 0.966
age by gender:
  0: 0.611
  1: 0.616
age by race:
  0: 0.659
  1: 0.598
  2: 0.612
  3: 0